In [ ]:
# Run this cell before the panel cells.
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# Set BASE to the folder containing these notebooks.
BASE = Path.cwd()
DATA_DIR = BASE / "data"
# PROJECT_ROOT contains the map and segment inputs.
PROJECT_ROOT = Path(os.environ.get("TOMTOM_PROJECT_ROOT", str(BASE / "source_data")))
OUTPUT_ROOT = BASE / "figures" / "fig1"
USE_SAVED_MAPS = True

def show_saved(path):
    with PILImage.open(path) as image:
        image = image.convert("RGB")
        width, height = image.size
        fig, ax = plt.subplots(figsize=(9, 9 * height / width))
        ax.imshow(image)
        ax.axis("off")
        fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
        plt.show()
        plt.close(fig)


In [ ]:
# Panel a: City map and count inset
def city_map(data_dir, project_root, output_dir):

    output_dir.mkdir(parents=True, exist_ok=True)
    import os
    import warnings
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    warnings.filterwarnings("ignore")

    FIG_W = 13.5
    FIG_H = 5.8
    DPI_SHOW = 160
    DPI_SAVE = 600

    FONT_FAMILY = "Arial"
    BASE_FONT_SIZE = 9
    TITLE_SIZE = 11
    TICK_SIZE = 8
    LEGEND_SIZE = 8

    SIZE_SMALL = 9
    SIZE_LARGE = 17
    SIZE_VERY_LARGE = 28
    SIZE_MEGA = 52

    CITY_COLORS = {
        "Small": "#4F566E",
        "Large": "#8C9EB4",
        "VeryLarge": "#F4C430",
        "Mega": "#CC5438"
    }

    LAND_FACE = "#EEEEEE"
    LAND_EDGE = "#B7B7B7"
    LAND_LINEWIDTH = 0.28

    POINT_EDGE = "white"
    POINT_ALPHA = 0.88
    POINT_LW = 0.35

    SPINE_COLOR = "#4A4A4A"
    SPINE_LW = 0.5

    MAP_TITLE = "Global distribution of study cities"

    XMIN, XMAX = -180, 180
    YMIN, YMAX = -57, 72

    XTICKS = np.arange(-180, 181, 60)
    YTICKS = np.arange(-45, 61, 15)

    SHP_PATH = str(project_root / 'GIS/world-administrative-boundaries/world-administrative-boundaries.shp')
    CSV_PATH = data_dir / "city_annual.csv"
    OUT_DIR = output_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    OUT_PNG = os.path.join(OUT_DIR, "global_city_distribution_cropped.png")
    OUT_PDF = os.path.join(OUT_DIR, "global_city_distribution_cropped.pdf")

    world = gpd.read_file(SHP_PATH)
    df = pd.read_csv(CSV_PATH)

    need_cols = ["longitude", "latitude", "population"]
    missing = [c for c in need_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing CSV columns: {missing}")

    for c in need_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=need_cols).copy()
    df = df[
        (df["longitude"].between(-180, 180)) &
        (df["latitude"].between(-90, 90)) &
        (df["population"] > 0)
    ].copy()

    def classify_city_size(population):
        if population < 1_000_000:
            return "Small"
        elif population < 5_000_000:
            return "Large"
        elif population < 10_000_000:
            return "VeryLarge"
        else:
            return "Mega"

    df["size_class"] = df["population"].apply(classify_city_size)

    size_map = {
        "Small": SIZE_SMALL,
        "Large": SIZE_LARGE,
        "VeryLarge": SIZE_VERY_LARGE,
        "Mega": SIZE_MEGA
    }

    draw_order = ["Mega", "VeryLarge", "Large", "Small"]
    legend_order = ["Small", "Large", "VeryLarge", "Mega"]

    legend_labels = {
        "Small": "Small cities (<1M)",
        "Large": "Large cities (1–5M)",
        "VeryLarge": "Very large cities (5–10M)",
        "Mega": "Mega cities (≥10M)"
    }

    df["plot_size"] = df["size_class"].map(size_map)

    gdf_pts = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326"
    )

    if world.crs is None:
        world = world.set_crs("EPSG:4326")
    else:
        world = world.to_crs("EPSG:4326")

    world_crop = world.cx[XMIN:XMAX, YMIN:YMAX].copy()

    gdf_pts = gdf_pts[
        gdf_pts["longitude"].between(XMIN, XMAX) &
        gdf_pts["latitude"].between(YMIN, YMAX)
    ].copy()

    plt.rcParams["font.family"] = FONT_FAMILY
    plt.rcParams["font.size"] = BASE_FONT_SIZE

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), dpi=DPI_SHOW)

    world_crop.plot(
        ax=ax,
        color=LAND_FACE,
        edgecolor=LAND_EDGE,
        linewidth=LAND_LINEWIDTH,
        zorder=1
    )

    for cls in draw_order:
        sub = gdf_pts[gdf_pts["size_class"] == cls]
        if len(sub) == 0:
            continue

        ax.scatter(
            sub["longitude"],
            sub["latitude"],
            s=sub["plot_size"],
            facecolor=CITY_COLORS[cls],
            edgecolor=POINT_EDGE,
            linewidth=POINT_LW,
            alpha=POINT_ALPHA,
            zorder=3 if cls == "Mega" else 2.8
        )

    ax.set_xlim(XMIN, XMAX)
    ax.set_ylim(YMIN, YMAX)

    ax.set_aspect("auto")

    ax.set_xlabel("Longitude", fontsize=BASE_FONT_SIZE + 1)
    ax.set_ylabel("Latitude", fontsize=BASE_FONT_SIZE + 1)

    ax.set_xticks(XTICKS)
    ax.set_yticks(YTICKS)
    ax.tick_params(axis="both", labelsize=TICK_SIZE, length=3.0, width=0.7, color=SPINE_COLOR)

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(SPINE_LW)
        spine.set_color(SPINE_COLOR)

    ax.set_title(MAP_TITLE, fontsize=TITLE_SIZE, pad=6)

    legend_elements = []
    for cls in legend_order:
        legend_elements.append(
            Line2D(
                [0], [0],
                marker="o",
                color="none",
                label=legend_labels[cls],
                markerfacecolor=CITY_COLORS[cls],
                markeredgecolor=POINT_EDGE,
                markeredgewidth=POINT_LW,
                markersize=np.sqrt(size_map[cls])
            )
        )

    leg = ax.legend(
        handles=legend_elements,
        loc="lower left",
        frameon=True,
        fontsize=LEGEND_SIZE,
        title="city size",
        title_fontsize=LEGEND_SIZE,
        borderpad=0.55,
        handletextpad=0.55,
        labelspacing=0.5
    )

    leg.get_frame().set_facecolor("white")
    leg.get_frame().set_edgecolor("#B0B0B0")
    leg.get_frame().set_linewidth(0.6)
    leg.get_frame().set_alpha(0.95)

    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=DPI_SAVE, bbox_inches="tight")
    fig.savefig(OUT_PDF, dpi=DPI_SAVE, bbox_inches="tight")

    print(f"Saved PNG: {OUT_PNG}")
    print(f"Saved PDF: {OUT_PDF}")

    pass

def city_counts(data_dir, project_root, output_dir):

    output_dir.mkdir(parents=True, exist_ok=True)
    import os
    from pathlib import Path
    import pandas as pd
    import matplotlib.pyplot as plt

    FIG_W = 4.6
    FIG_H = 4.0
    DPI_SHOW = 140
    DPI_SAVE = 600

    FONT_FAMILY = "Arial"
    BASE_FONT_SIZE = 12
    TITLE_SIZE = 12
    LABEL_SIZE = 12
    TICK_SIZE = 12
    ANNOT_SIZE = 12

    BAR_THICKNESS = 0.55

    BAR_COLORS = {
        "Small-medium cities": "#4F566E",
        "Large cities": "#8C9EB4",
        "Very large cities": "#F4C430",
        "Mega cities": "#CC5438"
    }

    TEXT_COLOR = "#2F2F2F"
    SPINE_COLOR = "#4A4A4A"
    GRID_COLOR = "#B8B8B8"

    SPINE_LW = 0.8
    GRID_LW = 0.7

    GRID_ALPHA = 0.28
    GRID_LS = "--"

    TITLE = "city counts by population class"
    X_LABEL = "Number of cities"
    Y_LABEL = "Population class"

    TEXT_OFFSET_FRAC = 0.02

    CSV_PATH = data_dir / "city_annual.csv"
    OUT_DIR = output_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    OUT_PNG = os.path.join(OUT_DIR, "city_count_by_population_class.png")
    OUT_PDF = os.path.join(OUT_DIR, "city_count_by_population_class.pdf")

    plt.rcParams.update({
        "figure.figsize": (FIG_W, FIG_H),
        "figure.dpi": DPI_SHOW,
        "savefig.dpi": DPI_SAVE,
        "font.family": FONT_FAMILY,
        "font.size": BASE_FONT_SIZE,
        "text.color": TEXT_COLOR,
        "axes.labelcolor": TEXT_COLOR,
        "xtick.color": TEXT_COLOR,
        "ytick.color": TEXT_COLOR,
    })

    df = pd.read_csv(CSV_PATH, encoding="utf-8")

    if "population" not in df.columns:
        raise ValueError("Missing CSV column: population.")

    df["population"] = pd.to_numeric(df["population"], errors="coerce")
    df = df[df["population"] > 0].copy()

    bins = [0, 1_000_000, 5_000_000, 10_000_000, float("inf")]
    labels = [
        "Small-medium cities",
        "Large cities",
        "Very large cities",
        "Mega cities"
    ]

    df["Pop_class"] = pd.cut(
        df["population"],
        bins=bins,
        labels=labels,
        right=False
    )

    class_counts = (
        df["Pop_class"]
        .value_counts()
        .reindex(labels)
        .fillna(0)
        .astype(int)
    )

    print(class_counts)

    fig, ax = plt.subplots()

    bar_colors = [BAR_COLORS[k] for k in class_counts.index]

    bars = ax.barh(
        class_counts.index,
        class_counts.values,
        color=bar_colors,
        edgecolor="none",
        height=BAR_THICKNESS
    )

    xmax = class_counts.max()
    offset = xmax * TEXT_OFFSET_FRAC

    for i, v in enumerate(class_counts.values):
        ax.text(
            v + offset,
            i,
            f"{v}",
            va="center",
            ha="left",
            fontsize=ANNOT_SIZE,
            color=TEXT_COLOR
        )

    ax.set_xlabel(X_LABEL, fontsize=LABEL_SIZE)
    ax.set_ylabel(Y_LABEL, fontsize=LABEL_SIZE)
    ax.set_title(TITLE, fontsize=TITLE_SIZE, pad=8)

    ax.grid(False)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(SPINE_LW)
    ax.spines["bottom"].set_linewidth(SPINE_LW)
    ax.spines["left"].set_color(SPINE_COLOR)
    ax.spines["bottom"].set_color(SPINE_COLOR)

    ax.tick_params(axis="x", labelsize=TICK_SIZE, length=3.5, width=0.8)
    ax.tick_params(axis="y", labelsize=TICK_SIZE, length=0)

    ax.invert_yaxis()

    ax.set_xlim(0, xmax * 1.16)

    plt.tight_layout()

    fig.savefig(OUT_PNG, bbox_inches="tight")
    fig.savefig(OUT_PDF, bbox_inches="tight")

    pass

    print(f"Saved PNG: {OUT_PNG}")
    print(f"Saved PDF: {OUT_PDF}")

output_dir = OUTPUT_ROOT / "a"
with plt.rc_context():
    city_counts(DATA_DIR, PROJECT_ROOT, output_dir)
plt.close("all")
basemap = PROJECT_ROOT / "GIS/world-administrative-boundaries/world-administrative-boundaries.shp"
saved_map = PROJECT_ROOT / "code/global_city_distribution_cropped.png"
if basemap.exists():
    with plt.rc_context():
        city_map(DATA_DIR, PROJECT_ROOT, output_dir)
    plt.close("all")
    show_saved(output_dir / "global_city_distribution_cropped.png")
elif USE_SAVED_MAPS and saved_map.exists():
    print("Displaying the saved map; boundary data are unavailable.")
    show_saved(saved_map)
else:
    print("Map skipped: provide the boundary shapefile.")
show_saved(output_dir / "city_count_by_population_class.png")


In [ ]:
# Panel b: Congested network length
def scaling_panels(data_dir, project_root, output_dir):

    output_dir.mkdir(parents=True, exist_ok=True)
    FIGSIZE = (6.4, 5.4)
    DPI_OUT        = 600
    FONT_FAMILY    = "Arial"
    OUT_FONT_SIZE  = 14
    IN_FONT_SIZE   = 11

    KEEP_PCT       = 0.95
    SCATTER_SIZE   = 18
    SCATTER_ALPHA  = 0.18
    LINE_ALPHA     = 0.90
    BAND_ALPHA     = 0.20

    TEXT_Y_START   = 0.96
    TEXT_LINE_GAP  = 0.06

    SPINE_COLOR    = "#4A4A4A"
    SPINE_LW       = 0.7

    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    import warnings

    warnings.filterwarnings("ignore", category=FutureWarning)
    plt.rcParams.update({"font.family": FONT_FAMILY, "font.size": OUT_FONT_SIZE})

    FILE_PATH = data_dir / "city_annual.csv"
    OUT_DIR = output_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    regions = ["Asia", "Europe", "North America", "South America", "Oceania"]
    palette = dict(zip(
        regions,
        ["tab:red", "tab:blue", "tab:purple", "tab:green", "tab:orange"]
    ))

    region_abbrev = {
        "Asia": "AS",
        "Europe": "EU",
        "North America": "NA",
        "South America": "SA",
        "Oceania": "OC"
    }

    X_COL = "population"
    EXCLUDE_AFRICA = True

    Y_COLS = ['congested_network_length']

    df = pd.read_csv(FILE_PATH)

    need_cols = [
        "city", "country", "continent", X_COL,
        "congested_network_length",
        "congestion_intensity",
        "congested_network_ratio"
    ]
    missing_cols = [c for c in need_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    df = df[need_cols].copy()

    for c in [X_COL, "congested_network_length", "congestion_intensity", "congested_network_ratio"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["continent"] = df["continent"].astype(str).str.strip()

    if EXCLUDE_AFRICA:
        df = df[df["continent"] != "Africa"].copy()

    df = df[df["continent"].isin(regions)].copy()
    df = df[df[X_COL] > 0].copy()

    df["congested_network_ratio"] = (
        df["congested_network_ratio"] * 100
    )

    def fmt_p(p):
        return "P < 0.001" if p < 0.001 else f"P = {p:.3f}"

    def pretty_title(y_col):
        title_map = {
            "congested_network_length": "Congestion road length",
            "congestion_intensity": "Congestion intensity",
            "congested_network_ratio": "Congestion extent",
        }
        return title_map.get(y_col, y_col)

    def pretty_ylabel(y_col):
        label_map = {
            "congested_network_length": "Congested network length (km·h)",
            "congestion_intensity": "Congestion intensity (min/km)",
            "congested_network_ratio": "Congested network ratio (%)",
        }
        return label_map.get(y_col, y_col)

    def output_stub(y_col):
        name_map = {
            "congested_network_length": "congestion_road_length",
            "congestion_intensity": "congestion_intensity",
            "congested_network_ratio": "congestion_extent",
        }
        return name_map.get(y_col, y_col)

    def fit_region_scaling(sub, x_col, y_col):
        X = sm.add_constant(np.log10(sub[x_col]))
        y = np.log10(sub[y_col])
        mdl = sm.OLS(y, X).fit()
        return mdl

    def get_beta_ci(model, alpha=0.05):
        beta = model.params.iloc[1]
        ci_lo, ci_hi = model.conf_int(alpha=alpha).iloc[1]
        p_val = model.pvalues.iloc[1]
        r2 = model.rsquared
        n = int(model.nobs)
        return beta, ci_lo, ci_hi, p_val, r2, n

    def plot_one_metric(y_col):
        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=120)
        stats_ordered = []

        for reg in regions:
            sub = df[df["continent"] == reg].copy()
            sub = sub.dropna(subset=[X_COL, y_col])
            sub = sub[(sub[X_COL] > 0) & (sub[y_col] > 0)].copy()

            if sub.empty:
                continue

            lo_x, hi_x = sub[X_COL].quantile([(1 - KEEP_PCT) / 2, 1 - (1 - KEEP_PCT) / 2])
            lo_y, hi_y = sub[y_col].quantile([(1 - KEEP_PCT) / 2, 1 - (1 - KEEP_PCT) / 2])

            sub = sub[
                sub[X_COL].between(lo_x, hi_x) &
                sub[y_col].between(lo_y, hi_y)
            ].copy()

            if len(sub) < 3:
                continue

            mdl = fit_region_scaling(sub, X_COL, y_col)
            beta, ci_lo_beta, ci_hi_beta, p_val, r2, n = get_beta_ci(mdl, alpha=0.05)
            stats_ordered.append((reg, beta, ci_lo_beta, ci_hi_beta, p_val, r2, n))

            x_line = np.logspace(
                np.log10(sub[X_COL].min()),
                np.log10(sub[X_COL].max()),
                300
            )
            preds = mdl.get_prediction(sm.add_constant(np.log10(x_line)))
            y_hat = 10 ** preds.predicted_mean
            ci_lo_line, ci_hi_line = preds.conf_int(alpha=0.05).T

            ax.fill_between(
                x_line, 10 ** ci_lo_line, 10 ** ci_hi_line,
                color=palette[reg], alpha=BAND_ALPHA
            )
            ax.plot(
                x_line, y_hat,
                color=palette[reg], lw=1.5, alpha=LINE_ALPHA
            )
            ax.scatter(
                sub[X_COL], sub[y_col],
                s=SCATTER_SIZE,
                alpha=SCATTER_ALPHA,
                color=palette[reg],
                edgecolor="none"
            )

            print(
                f"{y_col:<35} {reg:<13}: "
                f"β={beta:.2f} (95% CI {ci_lo_beta:.2f}–{ci_hi_beta:.2f}), "
                f"R²={r2:.2f}, {fmt_p(p_val)}, n={n}"
            )

        ax.set_xscale("log")
        if y_col == "congested_network_length":
            ax.set_yscale("log")
        else:
            ax.set_yscale("linear")

        ax.set_xlabel("Urban population", fontsize=OUT_FONT_SIZE)
        ax.set_ylabel(pretty_ylabel(y_col), fontsize=OUT_FONT_SIZE)
        ax.set_title(pretty_title(y_col), fontsize=OUT_FONT_SIZE + 2)

        ax.tick_params(
            axis="both",
            labelsize=OUT_FONT_SIZE,
            color=SPINE_COLOR,
            width=SPINE_LW
        )

        for side in ["top", "right", "left", "bottom"]:
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(SPINE_LW)
            ax.spines[side].set_color(SPINE_COLOR)

        for i, (reg, beta, ci_lo_beta, ci_hi_beta, p_val, r2, n) in enumerate(stats_ordered):
            y_txt = TEXT_Y_START - i * TEXT_LINE_GAP
            reg_short = region_abbrev.get(reg, reg)
            label = (
                f"{reg_short}: β = {beta:.2f}, "
                f"R² = {r2:.2f}, "
                f"{fmt_p(p_val)}"
            )
            ax.text(
                0.03, y_txt, label,
                transform=ax.transAxes,
                ha="left", va="top",
                color="black", fontsize=IN_FONT_SIZE
            )

        fig.tight_layout()

        out_png = os.path.join(OUT_DIR, f"{output_stub(y_col)}_by_continent_scaling.png")
        out_pdf = os.path.join(OUT_DIR, f"{output_stub(y_col)}_by_continent_scaling.pdf")
        fig.savefig(out_png, dpi=DPI_OUT, bbox_inches="tight")
        fig.savefig(out_pdf, dpi=DPI_OUT, bbox_inches="tight")

        print("✔ Saved:", out_png)
        print("✔ Saved:", out_pdf, "\n")

        pass

    if __name__ == "__main__":
        for y_col in Y_COLS:
            plot_one_metric(y_col)

output_dir = OUTPUT_ROOT / "b"
with plt.rc_context():
    scaling_panels(DATA_DIR, PROJECT_ROOT, output_dir)
plt.close("all")
show_saved(output_dir / 'congestion_road_length_by_continent_scaling.png')


In [ ]:
# Panel c: Congestion intensity
def scaling_panels(data_dir, project_root, output_dir):

    output_dir.mkdir(parents=True, exist_ok=True)
    FIGSIZE = (6.4, 5.4)
    DPI_OUT        = 600
    FONT_FAMILY    = "Arial"
    OUT_FONT_SIZE  = 14
    IN_FONT_SIZE   = 11

    KEEP_PCT       = 0.95
    SCATTER_SIZE   = 18
    SCATTER_ALPHA  = 0.18
    LINE_ALPHA     = 0.90
    BAND_ALPHA     = 0.20

    TEXT_Y_START   = 0.96
    TEXT_LINE_GAP  = 0.06

    SPINE_COLOR    = "#4A4A4A"
    SPINE_LW       = 0.7

    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    import warnings

    warnings.filterwarnings("ignore", category=FutureWarning)
    plt.rcParams.update({"font.family": FONT_FAMILY, "font.size": OUT_FONT_SIZE})

    FILE_PATH = data_dir / "city_annual.csv"
    OUT_DIR = output_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    regions = ["Asia", "Europe", "North America", "South America", "Oceania"]
    palette = dict(zip(
        regions,
        ["tab:red", "tab:blue", "tab:purple", "tab:green", "tab:orange"]
    ))

    region_abbrev = {
        "Asia": "AS",
        "Europe": "EU",
        "North America": "NA",
        "South America": "SA",
        "Oceania": "OC"
    }

    X_COL = "population"
    EXCLUDE_AFRICA = True

    Y_COLS = ['congestion_intensity']

    df = pd.read_csv(FILE_PATH)

    need_cols = [
        "city", "country", "continent", X_COL,
        "congested_network_length",
        "congestion_intensity",
        "congested_network_ratio"
    ]
    missing_cols = [c for c in need_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    df = df[need_cols].copy()

    for c in [X_COL, "congested_network_length", "congestion_intensity", "congested_network_ratio"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["continent"] = df["continent"].astype(str).str.strip()

    if EXCLUDE_AFRICA:
        df = df[df["continent"] != "Africa"].copy()

    df = df[df["continent"].isin(regions)].copy()
    df = df[df[X_COL] > 0].copy()

    df["congested_network_ratio"] = (
        df["congested_network_ratio"] * 100
    )

    def fmt_p(p):
        return "P < 0.001" if p < 0.001 else f"P = {p:.3f}"

    def pretty_title(y_col):
        title_map = {
            "congested_network_length": "Congestion road length",
            "congestion_intensity": "Congestion intensity",
            "congested_network_ratio": "Congestion extent",
        }
        return title_map.get(y_col, y_col)

    def pretty_ylabel(y_col):
        label_map = {
            "congested_network_length": "Congested network length (km)",
            "congestion_intensity": "Congestion intensity (min/km)",
            "congested_network_ratio": "Congested network ratio (%)",
        }
        return label_map.get(y_col, y_col)

    def output_stub(y_col):
        name_map = {
            "congested_network_length": "congestion_road_length",
            "congestion_intensity": "congestion_intensity",
            "congested_network_ratio": "congestion_extent",
        }
        return name_map.get(y_col, y_col)

    def fit_region_scaling(sub, x_col, y_col):
        X = sm.add_constant(np.log10(sub[x_col]))
        y = np.log10(sub[y_col])
        mdl = sm.OLS(y, X).fit()
        return mdl

    def get_beta_ci(model, alpha=0.05):
        beta = model.params.iloc[1]
        ci_lo, ci_hi = model.conf_int(alpha=alpha).iloc[1]
        p_val = model.pvalues.iloc[1]
        r2 = model.rsquared
        n = int(model.nobs)
        return beta, ci_lo, ci_hi, p_val, r2, n

    def plot_one_metric(y_col):
        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=120)
        stats_ordered = []

        for reg in regions:
            sub = df[df["continent"] == reg].copy()
            sub = sub.dropna(subset=[X_COL, y_col])
            sub = sub[(sub[X_COL] > 0) & (sub[y_col] > 0)].copy()

            if sub.empty:
                continue

            lo_x, hi_x = sub[X_COL].quantile([(1 - KEEP_PCT) / 2, 1 - (1 - KEEP_PCT) / 2])
            lo_y, hi_y = sub[y_col].quantile([(1 - KEEP_PCT) / 2, 1 - (1 - KEEP_PCT) / 2])

            sub = sub[
                sub[X_COL].between(lo_x, hi_x) &
                sub[y_col].between(lo_y, hi_y)
            ].copy()

            if len(sub) < 3:
                continue

            mdl = fit_region_scaling(sub, X_COL, y_col)
            beta, ci_lo_beta, ci_hi_beta, p_val, r2, n = get_beta_ci(mdl, alpha=0.05)
            stats_ordered.append((reg, beta, ci_lo_beta, ci_hi_beta, p_val, r2, n))

            x_line = np.logspace(
                np.log10(sub[X_COL].min()),
                np.log10(sub[X_COL].max()),
                300
            )
            preds = mdl.get_prediction(sm.add_constant(np.log10(x_line)))
            y_hat = 10 ** preds.predicted_mean
            ci_lo_line, ci_hi_line = preds.conf_int(alpha=0.05).T

            ax.fill_between(
                x_line, 10 ** ci_lo_line, 10 ** ci_hi_line,
                color=palette[reg], alpha=BAND_ALPHA
            )
            ax.plot(
                x_line, y_hat,
                color=palette[reg], lw=1.5, alpha=LINE_ALPHA
            )
            ax.scatter(
                sub[X_COL], sub[y_col],
                s=SCATTER_SIZE,
                alpha=SCATTER_ALPHA,
                color=palette[reg],
                edgecolor="none"
            )

            print(
                f"{y_col:<35} {reg:<13}: "
                f"β={beta:.2f} (95% CI {ci_lo_beta:.2f}–{ci_hi_beta:.2f}), "
                f"R²={r2:.2f}, {fmt_p(p_val)}, n={n}"
            )

        ax.set_xscale("log")
        if y_col == "congested_network_length":
            ax.set_yscale("log")
        else:
            ax.set_yscale("linear")

        ax.set_xlabel("Urban population", fontsize=OUT_FONT_SIZE)
        ax.set_ylabel(pretty_ylabel(y_col), fontsize=OUT_FONT_SIZE)
        ax.set_title(pretty_title(y_col), fontsize=OUT_FONT_SIZE + 2)

        ax.tick_params(
            axis="both",
            labelsize=OUT_FONT_SIZE,
            color=SPINE_COLOR,
            width=SPINE_LW
        )

        for side in ["top", "right", "left", "bottom"]:
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(SPINE_LW)
            ax.spines[side].set_color(SPINE_COLOR)

        for i, (reg, beta, ci_lo_beta, ci_hi_beta, p_val, r2, n) in enumerate(stats_ordered):
            y_txt = TEXT_Y_START - i * TEXT_LINE_GAP
            reg_short = region_abbrev.get(reg, reg)
            label = (
                f"{reg_short}: β = {beta:.2f}, "
                f"R² = {r2:.2f}, "
                f"{fmt_p(p_val)}"
            )
            ax.text(
                0.03, y_txt, label,
                transform=ax.transAxes,
                ha="left", va="top",
                color="black", fontsize=IN_FONT_SIZE
            )

        fig.tight_layout()

        out_png = os.path.join(OUT_DIR, f"{output_stub(y_col)}_by_continent_scaling.png")
        out_pdf = os.path.join(OUT_DIR, f"{output_stub(y_col)}_by_continent_scaling.pdf")
        fig.savefig(out_png, dpi=DPI_OUT, bbox_inches="tight")
        fig.savefig(out_pdf, dpi=DPI_OUT, bbox_inches="tight")

        print("✔ Saved:", out_png)
        print("✔ Saved:", out_pdf, "\n")

        pass

    if __name__ == "__main__":
        for y_col in Y_COLS:
            plot_one_metric(y_col)

output_dir = OUTPUT_ROOT / "c"
with plt.rc_context():
    scaling_panels(DATA_DIR, PROJECT_ROOT, output_dir)
plt.close("all")
show_saved(output_dir / 'congestion_intensity_by_continent_scaling.png')


In [ ]:
# Panel d: Congested network ratio
def scaling_panels(data_dir, project_root, output_dir):

    output_dir.mkdir(parents=True, exist_ok=True)
    FIGSIZE = (6.4, 5.4)
    DPI_OUT        = 600
    FONT_FAMILY    = "Arial"
    OUT_FONT_SIZE  = 14
    IN_FONT_SIZE   = 11

    KEEP_PCT       = 0.95
    SCATTER_SIZE   = 18
    SCATTER_ALPHA  = 0.18
    LINE_ALPHA     = 0.90
    BAND_ALPHA     = 0.20

    TEXT_Y_START   = 0.96
    TEXT_LINE_GAP  = 0.06

    SPINE_COLOR    = "#4A4A4A"
    SPINE_LW       = 0.7

    import os
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    import warnings

    warnings.filterwarnings("ignore", category=FutureWarning)
    plt.rcParams.update({"font.family": FONT_FAMILY, "font.size": OUT_FONT_SIZE})

    FILE_PATH = data_dir / "city_annual.csv"
    OUT_DIR = output_dir
    os.makedirs(OUT_DIR, exist_ok=True)

    regions = ["Asia", "Europe", "North America", "South America", "Oceania"]
    palette = dict(zip(
        regions,
        ["tab:red", "tab:blue", "tab:purple", "tab:green", "tab:orange"]
    ))

    region_abbrev = {
        "Asia": "AS",
        "Europe": "EU",
        "North America": "NA",
        "South America": "SA",
        "Oceania": "OC"
    }

    X_COL = "population"
    EXCLUDE_AFRICA = True

    Y_COLS = ['congested_network_ratio']

    df = pd.read_csv(FILE_PATH)

    need_cols = [
        "city", "country", "continent", X_COL,
        "congested_network_length",
        "congestion_intensity",
        "congested_network_ratio"
    ]
    missing_cols = [c for c in need_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    df = df[need_cols].copy()

    for c in [X_COL, "congested_network_length", "congestion_intensity", "congested_network_ratio"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["continent"] = df["continent"].astype(str).str.strip()

    if EXCLUDE_AFRICA:
        df = df[df["continent"] != "Africa"].copy()

    df = df[df["continent"].isin(regions)].copy()
    df = df[df[X_COL] > 0].copy()


    def fmt_p(p):
        return "P < 0.001" if p < 0.001 else f"P = {p:.3f}"

    def pretty_title(y_col):
        title_map = {
            "congested_network_length": "Congestion road length",
            "congestion_intensity": "Congestion intensity",
            "congested_network_ratio": "Congestion extent",
        }
        return title_map.get(y_col, y_col)

    def pretty_ylabel(y_col):
        label_map = {
            "congested_network_length": "Congested network length (km)",
            "congestion_intensity": "Congestion intensity (min/km)",
            "congested_network_ratio": "Congested network ratio (h/day)",
        }
        return label_map.get(y_col, y_col)

    def output_stub(y_col):
        name_map = {
            "congested_network_length": "congestion_road_length",
            "congestion_intensity": "congestion_intensity",
            "congested_network_ratio": "congestion_extent",
        }
        return name_map.get(y_col, y_col)

    def fit_region_scaling(sub, x_col, y_col):
        X = sm.add_constant(np.log10(sub[x_col]))
        y = np.log10(sub[y_col])
        mdl = sm.OLS(y, X).fit()
        return mdl

    def get_beta_ci(model, alpha=0.05):
        beta = model.params.iloc[1]
        ci_lo, ci_hi = model.conf_int(alpha=alpha).iloc[1]
        p_val = model.pvalues.iloc[1]
        r2 = model.rsquared
        n = int(model.nobs)
        return beta, ci_lo, ci_hi, p_val, r2, n

    def plot_one_metric(y_col):
        fig, ax = plt.subplots(figsize=FIGSIZE, dpi=120)
        stats_ordered = []

        for reg in regions:
            sub = df[df["continent"] == reg].copy()
            sub = sub.dropna(subset=[X_COL, y_col])
            sub = sub[(sub[X_COL] > 0) & (sub[y_col] > 0)].copy()

            if sub.empty:
                continue

            lo_x, hi_x = sub[X_COL].quantile([(1 - KEEP_PCT) / 2, 1 - (1 - KEEP_PCT) / 2])
            lo_y, hi_y = sub[y_col].quantile([(1 - KEEP_PCT) / 2, 1 - (1 - KEEP_PCT) / 2])

            sub = sub[
                sub[X_COL].between(lo_x, hi_x) &
                sub[y_col].between(lo_y, hi_y)
            ].copy()

            if len(sub) < 3:
                continue

            mdl = fit_region_scaling(sub, X_COL, y_col)
            beta, ci_lo_beta, ci_hi_beta, p_val, r2, n = get_beta_ci(mdl, alpha=0.05)
            stats_ordered.append((reg, beta, ci_lo_beta, ci_hi_beta, p_val, r2, n))

            x_line = np.logspace(
                np.log10(sub[X_COL].min()),
                np.log10(sub[X_COL].max()),
                300
            )
            preds = mdl.get_prediction(sm.add_constant(np.log10(x_line)))
            y_hat = 10 ** preds.predicted_mean
            ci_lo_line, ci_hi_line = preds.conf_int(alpha=0.05).T

            ax.fill_between(
                x_line, 10 ** ci_lo_line, 10 ** ci_hi_line,
                color=palette[reg], alpha=BAND_ALPHA
            )
            ax.plot(
                x_line, y_hat,
                color=palette[reg], lw=1.5, alpha=LINE_ALPHA
            )
            ax.scatter(
                sub[X_COL], sub[y_col],
                s=SCATTER_SIZE,
                alpha=SCATTER_ALPHA,
                color=palette[reg],
                edgecolor="none"
            )

            print(
                f"{y_col:<35} {reg:<13}: "
                f"β={beta:.2f} (95% CI {ci_lo_beta:.2f}–{ci_hi_beta:.2f}), "
                f"R²={r2:.2f}, {fmt_p(p_val)}, n={n}"
            )

        ax.set_xscale("log")
        if y_col == "congested_network_length":
            ax.set_yscale("log")
        else:
            ax.set_yscale("linear")

        ax.set_xlabel("Urban population", fontsize=OUT_FONT_SIZE)
        ax.set_ylabel(pretty_ylabel(y_col), fontsize=OUT_FONT_SIZE)
        ax.set_title(pretty_title(y_col), fontsize=OUT_FONT_SIZE + 2)

        ax.tick_params(
            axis="both",
            labelsize=OUT_FONT_SIZE,
            color=SPINE_COLOR,
            width=SPINE_LW
        )

        for side in ["top", "right", "left", "bottom"]:
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(SPINE_LW)
            ax.spines[side].set_color(SPINE_COLOR)

        for i, (reg, beta, ci_lo_beta, ci_hi_beta, p_val, r2, n) in enumerate(stats_ordered):
            y_txt = TEXT_Y_START - i * TEXT_LINE_GAP
            reg_short = region_abbrev.get(reg, reg)
            label = (
                f"{reg_short}: β = {beta:.2f}, "
                f"R² = {r2:.2f}, "
                f"{fmt_p(p_val)}"
            )
            ax.text(
                0.03, y_txt, label,
                transform=ax.transAxes,
                ha="left", va="top",
                color="black", fontsize=IN_FONT_SIZE
            )

        fig.tight_layout()

        out_png = os.path.join(OUT_DIR, f"{output_stub(y_col)}_by_continent_scaling.png")
        out_pdf = os.path.join(OUT_DIR, f"{output_stub(y_col)}_by_continent_scaling.pdf")
        fig.savefig(out_png, dpi=DPI_OUT, bbox_inches="tight")
        fig.savefig(out_pdf, dpi=DPI_OUT, bbox_inches="tight")

        print("✔ Saved:", out_png)
        print("✔ Saved:", out_pdf, "\n")

        pass

    if __name__ == "__main__":
        for y_col in Y_COLS:
            plot_one_metric(y_col)

output_dir = OUTPUT_ROOT / "d"
with plt.rc_context():
    scaling_panels(DATA_DIR, PROJECT_ROOT, output_dir)
plt.close("all")
show_saved(output_dir / 'congestion_extent_by_continent_scaling.png')
